# Tutorial 1: Manifold Basics for Swallowing Analysis

This notebook walks through the core concepts:
1. What a swallowing trajectory looks like as data
2. How to verify the low-dimensional manifold hypothesis
3. How to compute trajectory metrics (geodesic length, curvature, smoothness)
4. How metrics differ across dysphagia phenotypes

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA

from core.trajectory import SwallowingTrajectory
from core.metrics import extract_all_metrics, geodesic_length, mean_curvature
from core.manifold import SwallowingManifold
from simulation.trajectory_generator import SyntheticManifold, TrajectoryGenerator, DEFAULT_LANDMARK_GROUPS

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Generate Synthetic Swallowing Data

We create a 5-dimensional manifold embedded in 28D (14 landmarks × 2D), then generate trajectories for healthy and pathological swallowing.

In [ ]:
# Create manifold and generator
manifold = SyntheticManifold(intrinsic_dim=5, ambient_dim=28, seed=42)
gen = TrajectoryGenerator(manifold, n_frames=100, fps=25.0, seed=42)

# Generate one trajectory per condition
conditions = ['healthy', 'fibrotic', 'weak', 'compensatory', 'neurogenic']
examples = {c: gen.generate(condition=c, subject_id=f'example_{c}') for c in conditions}

# Inspect the healthy trajectory
h = examples['healthy']
print(f'Shape: {h.landmarks.shape} = {h.n_frames} frames × {h.n_dims} dimensions')
print(f'Duration: {h.time[-1]:.2f}s at {h.fps} fps')
print(f'Landmark groups: {list(DEFAULT_LANDMARK_GROUPS.keys())}')

## 2. Verify the Low-Dimensional Manifold Hypothesis

In [ ]:
# Generate a cohort for PCA
cohort = gen.generate_cohort(n_per_condition=30, conditions=conditions)
X = np.vstack([t.landmarks for t in cohort])

pca = PCA(n_components=15)
pca.fit(X)
cumvar = np.cumsum(pca.explained_variance_ratio_)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.bar(range(1, 16), pca.explained_variance_ratio_, color='steelblue')
ax1.axvline(5, color='red', ls='--', label='d=5')
ax1.set_xlabel('Component'); ax1.set_ylabel('Variance'); ax1.legend()
ax1.set_title('Eigenvalue Spectrum')

ax2.plot(range(1, 16), cumvar, 'o-', color='tomato')
ax2.axhline(0.99, color='gray', ls='--', label='99%')
ax2.axvline(5, color='red', ls='--', label='d=5')
ax2.set_xlabel('Components'); ax2.set_ylabel('Cumulative Variance')
ax2.legend(); ax2.set_title('Cumulative Variance')
plt.tight_layout(); plt.show()

print(f'\nd=5 captures {cumvar[4]:.1%} of variance from {X.shape[1]}D space')

## 3. Compute Trajectory Metrics

For each trajectory, we compute:
- **Geodesic length**: total path distance (effort)
- **Mean curvature**: coordination complexity
- **Peak velocity**: maximum motion speed
- **Smoothness**: jerk-based motion quality

In [ ]:
for cond, traj in examples.items():
    sm = traj.smooth()
    print(f'{cond:16s}  L={sm.arc_length():8.2f}  κ={np.mean(sm.curvature):8.4f}  '
          f'v_max={np.max(sm.speed):8.2f}  smooth={sm.smoothness_index():8.2f}')

## 4. Visualize Metric Distributions Across Phenotypes

In [ ]:
data = []
for traj in cohort:
    sm = traj.smooth()
    data.append({'condition': traj.condition, 'geodesic_length': sm.arc_length(),
                 'mean_curvature': np.mean(sm.curvature), 'peak_velocity': np.max(sm.speed)})
df = pd.DataFrame(data)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, col in zip(axes, ['geodesic_length', 'mean_curvature', 'peak_velocity']):
    sns.boxplot(data=df, x='condition', y=col, hue='condition', ax=ax, legend=False)
    ax.set_title(col.replace('_', ' ').title())
    ax.tick_params(axis='x', rotation=25)
plt.tight_layout(); plt.show()

## Next Steps

- **Notebook 2**: Full simulated experiment walkthrough (classification, SRVF, phases)
- **Notebook 3**: Real data analysis with clinical score comparison
- **Demo script**: `python demos/quick_start.py` for a complete automated run